# 1D J1-J2 inference: EuclideanRNN (seed 111-555)


This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
# For reproduction: adjust this path to make sure it's correct 
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [2]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  

def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e=False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")

    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [5]:
N=100
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'results_EuclideanRNN'

## J2=0.0

In [6]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251

In [11]:
seed=111
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-24.996599197387695-0j), var E = 0.006800000090152025
DMRG energy  is -44.1277
Time taken=0.072 hrs


In [12]:
seed=222
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.181499481201172-0.0010999999940395355j), var E = 0.032600000500679016
DMRG energy  is -44.1277
Time taken=0.052 hrs


In [13]:
seed=333
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-24.922199249267578+0.0006000000284984708j), var E = 0.6435999870300293
DMRG energy  is -44.1277
Time taken=0.051 hrs


In [14]:
seed=444
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.24920082092285+0.0013000000035390258j), var E = 0.2215999960899353
DMRG energy  is -44.1277
Time taken=0.049 hrs


In [15]:
seed=555
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-24.95009994506836+0.0003000000142492354j), var E = 0.09830000251531601
DMRG energy  is -44.1277
Time taken=0.049 hrs


## J2=0.2

In [16]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388

In [17]:
seed=111
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-19.97909927368164-0.00139999995008111j), var E = 0.34700000286102295
DMRG energy  is -40.7388
Time taken=0.08 hrs


In [18]:
seed=222
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-20.481199264526367-0.0010000000474974513j), var E = 0.03420000150799751
DMRG energy  is -40.7388
Time taken=0.057 hrs


In [19]:
seed=333
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-20.262699127197266-0.0003000000142492354j), var E = 0.27309998869895935
DMRG energy  is -40.7388
Time taken=0.06 hrs


In [20]:
seed=444
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-20.41360092163086-0.0010999999940395355j), var E = 0.08669999986886978
DMRG energy  is -40.7388
Time taken=0.056 hrs


In [21]:
seed=555
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-19.803499221801758+0.0017000000225380063j), var E = 0.489300012588501
DMRG energy  is -40.7388
Time taken=0.052 hrs


# J2=0.5

- Trained on a local machine

In [4]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=37.5000

In [11]:
seed=111
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/local_machine/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-13.065099716186523+0.0005000000237487257j), var E = 0.10719999670982361
DMRG energy  is 37.5
Time taken=0.028 hrs


In [9]:
seed=222
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/local_machine/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-13.243599891662598-0.0005000000237487257j), var E = 0.049300000071525574
DMRG energy  is 37.5
Time taken=0.036 hrs


In [7]:
# Best model saved at epoch 972 with best E=-35.64011+0.08772j, varE=2.97259
seed=333
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/local_machine/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-35.08570098876953+0.003800000064074993j), var E = 5.235799789428711
DMRG energy  is 37.5
Time taken=0.039 hrs


In [8]:
seed=444
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/local_machine/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-12.22760009765625+0.001500000013038516j), var E = 0.186599999666214
DMRG energy  is 37.5
Time taken=0.045 hrs


In [6]:
seed=555
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/local_machine/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-13.242600440979004-0.00039999998989515007j), var E = 0.03819999843835831
DMRG energy  is 37.5
Time taken=0.037 hrs


## J2=0.8

In [28]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643

In [29]:
seed=111
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-16.5575008392334+0.0019000000320374966j), var E = 15.128700256347656
DMRG energy  is -42.0701
Time taken=0.079 hrs


In [30]:
seed=222
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-5.994200229644775-0.00039999998989515007j), var E = 0.21279999613761902
DMRG energy  is -42.0701
Time taken=0.053 hrs


In [31]:
seed=333
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-24.578399658203125-0.008999999612569809j), var E = 19.027999877929688
DMRG energy  is -42.0701
Time taken=0.078 hrs


In [32]:
seed=444
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-20.181499481201172+0j), var E = 0.05909999832510948
DMRG energy  is -42.0701
Time taken=0.081 hrs


In [33]:
seed=555
eRNN_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
total_params = sum(p.numel() for p in eRNN_00.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclRNN_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-20.12849998474121-0.0008999999845400453j), var E = 0.25540000200271606
DMRG energy  is -42.0701
Time taken=0.08 hrs
